In [96]:
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops
from skimage.measure import shannon_entropy
import matplotlib.pyplot as plt
import os
import cv2

# Definir as variáveis de caminho
DATADIR = 'C:/Users/Usuário/Downloads/dataset/'
TABLEDIR = DATADIR+'data.csv'
IMAGEDIR = DATADIR+'images'

# Ler os valores tabulares
df = pd.read_csv(TABLEDIR)
# Verificar o número total de entradas
print("O número total de entradas é: ", df.shape[0], "\n")
# Verificar celulas vazias na tabela
print("O número de células vazias por coluna é: \n", df.isnull().sum())

O número total de entradas é:  33126 

O número de células vazias por coluna é: 
 image_name                         0
patient_id                         0
sex                               65
age_approx                        68
anatom_site_general_challenge    527
diagnosis                          0
benign_malignant                   0
target                             0
dtype: int64


In [97]:
# Como podemos ver, algumas das células não possuem valor. A coluna com a maior concentração de células vazias é anatom_site_general_challenge
# A quantidade de células vazias corresponde a 1.59% das entradas.
uniqueAnatom = df["anatom_site_general_challenge"].value_counts(dropna=False)
print(uniqueAnatom, "\n")

# Para não perdermos amostras, as células vazias categóricas (anatom e sex) serão preenchidas como 'unknown', enquanto as células vazias contínuas (age) serão preemchidas com a média
mean_age = df["age_approx"].mean()
df["age_approx"] = df["age_approx"].fillna(mean_age)
df = (df.fillna("unknown"))
print("O número de células vazias por coluna é: \n", df.isnull().sum())

anatom_site_general_challenge
torso              16845
lower extremity     8417
upper extremity     4983
head/neck           1855
NaN                  527
palms/soles          375
oral/genital         124
Name: count, dtype: int64 

O número de células vazias por coluna é: 
 image_name                       0
patient_id                       0
sex                              0
age_approx                       0
anatom_site_general_challenge    0
diagnosis                        0
benign_malignant                 0
target                           0
dtype: int64


In [98]:
# Podemos notar que algumas colunas possuem valores de texto, então iremos mapear esses campos em categorias, atribuindo um valor a cada uma delas
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["anatom_site_general_challenge"] = le.fit_transform(df["anatom_site_general_challenge"])
df["sex"] = le.fit_transform(df["sex"])
df["diagnosis"] = le.fit_transform(df["diagnosis"])
df["benign_malignant"] = le.fit_transform(df["benign_malignant"])

mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(mapping)

{'benign': np.int64(0), 'malignant': np.int64(1)}


In [100]:
# Campos mapeados anteriormente representam categorias, então não precisam ser normalizados.
# Já o campo de idade é representado na tabela como um valor contínuo, nós iremos normalizar ele

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

uniqueAnatom = df["age_approx"].value_counts(dropna=False)
print(uniqueAnatom, "\n")

df["age_approx"] = scaler.fit_transform(df[["age_approx"]])

uniqueAnatom = df["age_approx"].value_counts(dropna=False)
print(uniqueAnatom, "\n")



age_approx
45.000000    4466
50.000000    4270
55.000000    3824
40.000000    3576
60.000000    3240
35.000000    2850
65.000000    2527
30.000000    2358
70.000000    1968
25.000000    1544
75.000000     981
20.000000     655
80.000000     419
85.000000     149
15.000000     132
90.000000      80
48.870016      68
10.000000      17
0.000000        2
Name: count, dtype: int64 

age_approx
0.500000    4466
0.555556    4270
0.611111    3824
0.444444    3576
0.666667    3240
0.388889    2850
0.722222    2527
0.333333    2358
0.777778    1968
0.277778    1544
0.833333     981
0.222222     655
0.888889     419
0.944444     149
0.166667     132
1.000000      80
0.543000      68
0.111111      17
0.000000       2
Name: count, dtype: int64 

